# Sudoku Knowledge Representation & Inference

You will implement three functions -- `build_general_kb`, `build_definite_kb`, `pl_bc_entails` -- and the full-grid solving logic in `sudoku_solver.py`.

This notebook imports and tests those functions. The Streamlit app (`sudoku_app.py`) must import the same implementation from `sudoku_solver.py`; do not copy or rewrite the solver functions inside the app.

**Rules:**

- In `sudoku_solver.py`, import only from `utils.py` and `logic_.py`; do not modify either file.
- Do not duplicate the core solver functions in this notebook or `sudoku_app.py`.
- All other content in this notebook may be edited freely.


In [1]:
from utils import *
from logic_ import *
import json
import time
import importlib
import sudoku_solver

# Reload so edits made to sudoku_solver.py are picked up when this cell is rerun.
importlib.reload(sudoku_solver)

from sudoku_solver import (
    atom,
    build_general_kb,
    build_definite_kb,
    solve_full_grid_fc,
    pl_bc_entails,
    solve_full_grid_bc,
)


## Loading a puzzle from JSON

Puzzles are provided as JSON, not embedded in this notebook. Each file looks like:

```json
{
  "n": 9, "box_h": 3, "box_w": 3,
  "puzzles": [
    {
      "givens": {"1_1": 3, "2_3": 1, ...},
      "given_count": 28,
      "solution": {"1_1": 3, "1_2": 4, ...}
    },
    ...
  ]
}
```

`"r_c"` string keys map to the value at row `r`, column `c` (1-indexed). `given_count` is exactly how many cells are given. `solution` is included so you can check your own work as you go, but your functions must not read `solution` to answer a query, they should only read `givens`.

In [2]:
#do not change this function, it is used to load the puzzle pool from a json file
def load_pool(path):
    with open(path) as f:
        raw = json.load(f)
    puzzles = []
    for p in raw['puzzles']:
        givens = {tuple(int(x) for x in k.split('_')): v for k, v in p['givens'].items()}
        solution = {tuple(int(x) for x in k.split('_')): v for k, v in p['solution'].items()}
        puzzles.append({'givens': givens, 'solution': solution, 'given_count': p['given_count']})
    return raw['n'], raw['box_h'], raw['box_w'], puzzles

n, box_h, box_w, puzzle_pool = load_pool('puzzles.json')
print(f'{len(puzzle_pool)} puzzles loaded, {n}x{n} grid, {box_h}x{box_w} boxes')
print('given_count values:', sorted(p['given_count'] for p in puzzle_pool))

# Pick one puzzle to work with through the rest of this notebook.
puzzle = puzzle_pool[0]
givens = puzzle['givens']
print(f"working puzzle has {puzzle['given_count']} givens")
for r in range(1, n + 1):
    print([givens.get((r, c), '.') for c in range(1, n + 1)])

5 puzzles loaded, 9x9 grid, 3x3 boxes
given_count values: [30, 33, 36, 39, 42]
working puzzle has 30 givens
['.', 3, '.', '.', '.', '.', '.', '.', '.']
[2, '.', '.', '.', '.', 7, 8, 6, '.']
[5, 8, '.', 2, 6, '.', '.', 3, '.']
[7, 5, '.', '.', '.', '.', '.', 8, '.']
['.', '.', '.', '.', 7, '.', 5, '.', 4]
['.', '.', '.', 5, 3, '.', '.', 9, 6]
['.', 1, 2, '.', '.', 9, '.', '.', '.']
[6, 4, '.', '.', 5, 8, 9, '.', '.']
['.', '.', '.', '.', 2, 3, '.', '.', '.']


Notice: `pl_fc_entails`/`pl_resolution`/`tt_entails` are all given to you, already implemented, in `logic_.py`. The one algorithm you write yourself in this assignment is **backward chaining** (`pl_bc_entails`) -- see Task 2.

## Define Symbols

Two families of propositional symbols, for row $r$, column $c$, value $v$ (all ranging over $1$ to $n$):

| Symbol | Meaning |
|---|---|
| $\mathit{Is}_{rcv}$ | cell $(r,c)$ has value $v$ |
| $\mathit{Not}_{rcv}$ | cell $(r,c)$ does **not** have value $v$ |

For each representation in Task 1, determine which of these two families is actually needed.

Hint: consider what form a definite (Horn) clause must take, and what `PropDefiniteKB.tell()` will accept.

### Helper function

`atom(prefix, r, c, v)`, where prefix can be either `Is` or `Not`, is a naming helper so propositional symbols need not be typed. It is provided for you in `sudoku_solver.py`; do not change it.

In code, $\mathit{Is}_{rcv}$ and $\mathit{Not}_{rcv}$ are written as single-word symbol names, e.g. `Is3_2_4` and `Not3_2_4`: the prefix is followed immediately by `r`, then by `c` and `v` separated by underscores. No separator is needed between the prefix and `r` since `expr()` only requires a symbol name to start with an uppercase letter. So `Is3_2_4` is parsed as one valid symbol.


In [3]:
# atom() is provided in sudoku_solver.py and imported above.


## Part A: Design the Sudoku Solver

### A.1) Knowledge Representation - Build KB

Every well-posed Sudoku puzzle satisfies exactly these conditions:

- Each cell is assigned **at least one** value from $\{1, \dots, n\}$.
- Each cell is assigned **at most one** value from $\{1, \dots, n\}$, i.e., it cannot hold two different values at once.
- No two cells in the same row hold the same value.
- No two cells in the same column hold the same value.
- No two cells in the same box hold the same value.
- The **givens** cells hold their stated values.

Formalize these Sudoku constraints as propositional logic, in **two** representations:

**(a) General clauses -  `build_general_kb`.**

Encode each of the following directly: no restriction here; you may use arbitrary disjunctions of positive or negated literals. Must return a `PropKB`. 

**(b) Definite (Horn) clauses:- `build_definite_kb`.** Must return a `PropDefiniteKB`. Recall a definite clause is a disjunction of literals with exactly one *positive* literal.

Equivalently written as an implication whose conclusion is a single positive literal and whose premises are a conjunction of positive literals: `P1 & P2 & ... & Pk ==> Q`.

`PropDefiniteKB.tell()` will reject anything else.

Both functions take the puzzle's givens as fixed facts.

- **Implement `build_general_kb()` and `build_definite_kb()` in sudoku_solver.py**
- **Rerun the import cell near the top of this notebook after making changes.**
- **Explain your representation in Conceptual Question 1 below.**

In [4]:
# Implement build_general_kb() and build_definite_kb() in sudoku_solver.py.
# Rerun the import cell near the top of this notebook after making changes.


### A.2) Solve the Puzzle

**(a) Resolution and model checking on the general representation.** Using `build_general_kb` and the library's `pl_resolution` / `tt_entails`, try to solve the puzzle, i.e., for each cell, determine which value is entailed. Attempt this in the code cell below: it is provided commented out, because both are sound and complete on `build_general_kb`'s output but neither scales to the full grid, and it is expected to hang or take an impractically long time. Uncomment a few lines at a time and give each at most about 30 seconds; use Kernel > Interrupt if it hasn't returned by then, and note what you observed.

Explain in your own words: what specifically makes `pl_resolution`'s cost grow out of control here, and separately, what makes `tt_entails`'s cost grow out of control? A simple complexity argument for each is the expected answer.

**Use your observations in Conceptual Question 2 below.**


In [5]:
# Attempt (a): try solving the puzzle using pl_resolution / tt_entails on the
# general KB. Left commented out because it is expected to take an
# impractically long time (pl_resolution) or be outright infeasible
# (tt_entails) on the full grid.
#
# Uncomment a few lines at a time and give each at most ~30 seconds; use
# Kernel > Interrupt if it hasn't returned by then.
#
general_kb = build_general_kb(n, box_h, box_w, givens)
r, c, v = 1, 1, givens.get((1, 1), 1)  # try one cell/value pair
query = atom('Is', r, c, v)
#
# print(pl_resolution(general_kb, query))    
# print(tt_entails(associate('&', general_kb.clauses), query)) 

### Observation notes

Building the general KB completes, but attempting a full-grid entailment run is not practical. Even a single query causes resolution to create and compare a rapidly growing set of resolvents; it did not finish within the suggested short interactive budget. Truth-table enumeration is even less feasible: the direct encoding has 729 `Is` symbols, so a naive complete model check has up to $2^{729}$ truth assignments. I therefore interrupted/left these exhaustive calls disabled and used the Horn-clause algorithms for the full grid.


**(b) Forward chaining on the full grid --** Implement `solve_full_grid_fc()` in sudoku_solver.py. Using your above `build_definite_kb` and the library's `pl_fc_entails` (no need to reimplement them), solve the whole puzzle. Find the value that's entailed for every cell. Reconstruct and display the solved grid.

**(c) Backward chaining -- implement it yourself.**
```python
pl_bc_entails(kb, query) -> bool
```
Implement `pl_bc_entails()` in sudoku_solver.py. Start from the query and recursively try to prove each premise of a rule whose conclusion matches the current goal, bottoming out at known facts. Your function must agree with `pl_fc_entails` on every cell/value pair in this puzzle including correctly returning `False` for values that are *not* part of the solution.

**(d) Backward chaining on the full grid --** Implement `solve_full_grid_bc` in sudoku_solver.py. Using your above `build_definite_kb` and your own `pl_bc_entails`, solve the whole puzzle the same way `solve_full_grid_fc` does: for every cell, try each candidate value until `pl_bc_entails` confirms one. Time both `solve_full_grid_fc` and `solve_full_grid_bc` on the same puzzle and compare. Use your measured result where relevant in Conceptual Question 5.


In [6]:
# Implement solve_full_grid_fc(), pl_bc_entails(), and solve_full_grid_bc()
# in sudoku_solver.py. Rerun the import cell near the top after making changes.


### Verifying the algorithms

Uncomment the following block of code to validate your code.

In [7]:
# Validate recursive backward chaining and benchmark both full-grid solvers
# on every supplied puzzle in the same local environment.
import statistics
import time

# Detailed completeness/soundness agreement check for the working puzzle.
definite_kb = build_definite_kb(n, box_h, box_w, givens)
for (r, c), correct_v in puzzle['solution'].items():
    for candidate_v in range(1, n + 1):
        actual = pl_bc_entails(
            definite_kb,
            atom('Is', r, c, candidate_v),
        )
        assert actual == (candidate_v == correct_v)

# Full-grid correctness and timing for all five puzzles.
timing_rows = []
for puzzle_number, test_puzzle in enumerate(puzzle_pool, 1):
    test_givens = test_puzzle['givens']
    expected = test_puzzle['solution']

    started = time.perf_counter()
    solved_fc = solve_full_grid_fc(n, box_h, box_w, test_givens)
    fc_elapsed = time.perf_counter() - started
    assert solved_fc == expected

    started = time.perf_counter()
    solved_bc = solve_full_grid_bc(n, box_h, box_w, test_givens)
    bc_elapsed = time.perf_counter() - started
    assert solved_bc == expected

    timing_rows.append((
        puzzle_number,
        test_puzzle['given_count'],
        fc_elapsed,
        bc_elapsed,
    ))

print('Puzzle  Givens  FC (s)  BC (s)  Cells solved')
for puzzle_number, given_count, fc_elapsed, bc_elapsed in timing_rows:
    print(
        f'{puzzle_number:>6}  {given_count:>6}  '
        f'{fc_elapsed:>6.2f}  {bc_elapsed:>6.2f}  81/81 by both'
    )

fc_time = statistics.mean(row[2] for row in timing_rows)
bc_time = statistics.mean(row[3] for row in timing_rows)
print(f'Average          {fc_time:>6.2f}  {bc_time:>6.2f}')
print('All recursive-BC truth checks and all five full-grid checks passed.')


Puzzle  Givens  FC (s)  BC (s)  Cells solved
     1      30    0.57    2.72  81/81 by both
     2      33    0.63    2.60  81/81 by both
     3      36    0.57    2.60  81/81 by both
     4      39    0.55    2.59  81/81 by both
     5      42    0.54    2.57  81/81 by both
Average            0.57    2.62
All recursive-BC truth checks and all five full-grid checks passed.


## Part B: Conceptual Questions

Answer all five questions directly in this notebook. Replace each **Your answer:** placeholder with your own response.


### 1. Detailed Representation Strategy: General vs. Definite (Horn) Encoding

Explain in detail how you formalized the Sudoku puzzle constraints into propositional logic across both Knowledge Base representations:

**(a) General KB Strategy (`build_general_kb`):** Detail how standard Sudoku rules (e.g., at-least-one value per cell, at-most-one value per cell, row/column/box uniqueness) are directly translated into Conjunctive Normal Form (CNF) clauses without structural restrictions.

**(b) Definite KB Strategy (`build_definite_kb`):** Definite/Horn clauses strictly permit at most one positive literal per clause, prohibiting disjunctive constraints like $(Is_{r,c,1} \lor Is_{r,c,2} \lor Is_{r,c,3} \lor ... \lor Is_{r,c,n})$. Explain step-by-step how your encoding deals with this issue.


**Your answer:**

**(a) General KB.** I used only the $Is_{r,c,v}$ atoms. For every cell $(r,c)$, the clause $Is_{r,c,1} \lor \cdots \lor Is_{r,c,n}$ enforces at least one value. For every pair $v_1<v_2$, the binary clause $\neg Is_{r,c,v_1} \lor \neg Is_{r,c,v_2}$ enforces at most one value. Row uniqueness is encoded by $\neg Is_{r,c_1,v} \lor \neg Is_{r,c_2,v}$ for every pair of columns in a row; column uniqueness is analogous for row pairs; and box uniqueness uses the same binary clause for every pair of cells in a box. Each given is a positive unit clause $Is_{r,c,v}$. All of these are already CNF clauses, and `PropKB.tell` stores their conjunction. Together, at-least-one plus the three at-most-one families also force every row, column, and box to contain every digit exactly once on an $n \times n$ grid.

**(b) Definite KB.** A definite clause cannot express the multi-positive at-least-one disjunction directly, and it also cannot use a negated atom as its single conclusion. I therefore reified negative information as positive symbols $Not_{r,c,v}$. A fact $Is_{r,c,v}$ implies: (1) $Not_{r,c,u}$ for each other value $u$ in the same cell, and (2) $Not_{r',c',v}$ for every row, column, or box peer $(r',c')$. Finally, for each candidate $v$, the conjunction of all other eliminations implies the value: $\bigwedge_{u\ne v} Not_{r,c,u} \Rightarrow Is_{r,c,v}$. Givens are positive `Is` facts. Thus every stored statement is either a fact or an implication with positive premises and one positive conclusion. The encoding supports repeated elimination and last-candidate deductions without violating Horn syntax.


### 2. Theoretical Completeness vs. Computational Tractability

Model checking and resolution-refutation are sound and complete—they are guaranteed to terminate with a correct answer for any propositional KB. Despite this guarantee, explain whether you would use either as the default algorithm for solving Sudoku puzzles. *(Hint: Consider space/time complexity and state-space growth, and use your observations from the experiment above where relevant.)*


**Your answer:**

I would not use either algorithm as the default Sudoku solver. Truth-table entailment is $O(2^m)$ in the number $m$ of proposition symbols. The general 9-by-9 encoding has 729 `Is` symbols, so enumerating all assignments is astronomically large even though constraints reject most models. Resolution is complete, but it repeatedly resolves compatible clause pairs; the number and size of derived clauses can grow exponentially, and duplicate/subsumption checks add substantial work. In the experiment the general KB could be built, but a full entailment run was not usable within the suggested interactive time budget. Completeness guarantees the eventual answer, not a practical answer time or memory footprint. Horn forward/backward chaining exploits the restricted rule form and is polynomial in the KB size, making it the appropriate default for the elimination-solvable puzzles supplied here.


### 3. Backward Chaining: Design, Pseudocode, and Challenges

Write pseudocode for `pl_bc_entails(kb, query)`, the backward-chaining algorithm you implemented. Show, at a level of detail that reveals the algorithm's structure (not full Python), how the function checks whether the query is already a known fact, finds candidate rules whose conclusion matches the current goal, recursively proves each premise of such a rule, and combines results—both across the premises of one rule and across multiple candidate rules—to reach a single boolean answer.

Then, in your own words, discuss the design challenges you had to work through to make your algorithm both correct and guaranteed to terminate on every puzzle, and explain how your pseudocode addresses them.


**Your answer:**

The current `pl_bc_entails` solution uses recursive, goal-directed proof search. It first builds reusable indexes containing all known facts and a mapping from each possible conclusion to the premise lists of rules that can establish it. Proven propositions are cached on the knowledge base so later cell/candidate queries can reuse successful work.

```text
BC-ENTAILS(KB, query):
    facts <- all fact clauses in KB
    rules[head] <- all premise lists whose rule conclusion is head
    proved <- cached facts and successfully proved goals

    repeat:
        failed <- empty set
        proved_count_before <- size(proved)

        PROVE(goal, visiting):
            if goal in proved: return True
            if goal in failed: return False
            if goal in visiting: return False

            add goal to visiting

            for each premises in rules[goal]:       # OR across rules
                rule_succeeds <- True
                for each premise in premises:       # AND within one rule
                    if not PROVE(premise, visiting):
                        rule_succeeds <- False
                        break

                if rule_succeeds:
                    remove goal from visiting
                    add goal to proved
                    record the successful rule if tracing is enabled
                    return True

            remove goal from visiting
            add goal to failed
            return False

        if PROVE(query, empty set): return True
        if size(proved) == proved_count_before: return False
```

`PROVE` starts with the requested query. A goal succeeds immediately when it is a fact or has already been proved. Otherwise, the function retrieves only rules whose conclusion matches that goal. The rules are alternatives, so one successful rule is enough (OR). Within a selected rule, every premise must be recursively proved (AND). When all premises succeed, the conclusion is added to `proved` and can support later deductions and later calls.

Cycle handling is necessary because the Sudoku KB connects `Is` propositions to `Not` eliminations and combines several `Not` propositions to derive another `Is` value. The `visiting` set contains only goals on the active recursive path. Encountering a goal already in that set rejects that cyclic branch while allowing the search to try other matching rules. The `failed` set prevents the same unsuccessful subproblem from being expanded repeatedly during one pass.

A recursive call can prove useful intermediate propositions even when the requested query does not succeed during that pass. Therefore, the outer loop starts a new pass whenever the shared `proved` set has grown. It returns `False` only when a complete pass proves neither the query nor any new proposition. Termination is guaranteed because the knowledge base contains finitely many propositions and each successful proposition is added to `proved` at most once. The cached fact/rule indexes and `proved` set also reduce repeated work when `solve_full_grid_bc` tests many candidate values.


### 4. Expressive Limits of Horn Logic

Named elimination techniques such as Naked Pairs and X-Wing can, in fact, be encoded as definite clauses, using the same `Is`/`Not` vocabulary as your `build_definite_kb`. Work out how you would encode one of these techniques as definite clauses, and discuss the consequences of doing so.


**Your answer:**

I would encode a **Naked Pair**. Suppose cells $a$ and $b$ are in the same unit (row, column, or box), and their only remaining candidates are $x$ and $y$. With the `Not` vocabulary, that condition is the positive conjunction $P=\bigwedge_{k\notin\{x,y\}}(Not_{a,k} \land Not_{b,k})$. The ordinary Sudoku constraints already ensure that each cell takes a value and that two cells in one unit cannot take the same value. Therefore $a$ and $b$ must occupy $x$ and $y$ in some order. For every other cell $t$ in the unit, add two definite clauses: $P \Rightarrow Not_{t,x}$ and $P \Rightarrow Not_{t,y}$. Each clause has only positive premises and one positive conclusion, so it is definite.

The benefit is stronger propagation: puzzles that stall under single-candidate elimination may become solvable without search. The cost is combinatorial rule growth. The encoder must enumerate units, cell pairs, value pairs, and each other target cell; analogous X-Wing rules enumerate row/column and candidate patterns and are larger still. The rules also make traces and debugging harder, and every pattern must be encoded carefully to remain sound. Horn logic is therefore expressive enough, but the representation can become unwieldy compared with a CSP solver that detects such patterns procedurally.


### 5. Data-Driven vs. Goal-Driven Performance

Forward chaining (data-driven) and backward chaining (goal-driven) are both sound and complete for Horn KBs, but their execution runtimes vary depending on the target query.

**(a)** Describe a scenario—in terms of total KB size versus the query-relevant subset—where backward chaining is significantly faster than forward chaining.

**(b)** Describe a scenario where backward chaining offers no performance advantage, or performs worse than forward chaining.

Use your measured forward- vs. backward-chaining result where relevant.


**Your answer:**

**(a)** Backward chaining is attractive for one narrow query in a very large KB when only a small ancestor subgraph can support it. For example, if a requested cell/value depends on a short chain rooted in a few nearby givens while most rules concern unrelated regions, recursive BC explores only matching rules and their premises. Forward chaining still propagates every initial fact through all reachable rules before it can finish.

**(b)** Backward chaining offers little advantage when the query-relevant dependency graph covers most of the KB. That is the situation for a full Sudoku grid: row, column, and box rules connect many `Is` and `Not` propositions, and `solve_full_grid_bc` launches separate candidate queries for many cells. Forward chaining computes one closure and reuses it for the entire solution; BC repeatedly pays recursive goal-selection and branch-exploration costs even though its successful-result cache helps.

In the executed five-puzzle benchmark above, both algorithms solved all 81 cells correctly for every puzzle, but FC consistently completed much faster than recursive BC. Exact seconds vary by machine and run, so the table's same-environment measurements—not Streamlit Cloud timings—are the appropriate comparison. The result supports the structural explanation: goal direction can save work for a selective single query, but it is a disadvantage when almost the whole connected rule graph is needed for a full-grid solution. False queries can be especially expensive because BC must exhaust all matching proof alternatives before returning `False`.


## Part C: Streamlit Integration

Wrap your Sudoku solver and inference logic into an interactive Streamlit application, `sudoku_app.py`. Your application must implement the following:

**1. Puzzle selection & visual board display**
- An interactive selector/dropdown to pick any puzzle from `puzzles.json`.
- A visual rendering of the grid that clearly distinguishes the initial *givens* from empty cells.

**2. Full-grid auto-solver, with algorithm selection**
- A control (e.g. radio buttons) letting the user choose forward chaining (`solve_full_grid_fc`) or backward chaining (`solve_full_grid_bc`) before solving.
- A button that solves the full grid with the chosen algorithm, renders the solved state, and displays how long the solve took -- so the timing gap from task (d) is visible in the app, not just in the notebook.

**3. Targeted cell entailment query**
- Inputs for row ($r$), column ($c$), and value ($v$).
- A button that checks whether $\mathit{Is}_{rcv}$ is entailed (using `pl_bc_entails`) and displays the boolean verdict (`True` / `False`).

**4. Reasoning trace ("tutor mode")**
- Instrument your chosen inference algorithm (forward or backward chaining) to record the reasoning steps it takes while answering a query.
- Present that trace in a human-readable format -- not a raw Python string or internal symbol dictionary. For example: expandable cards/accordions showing the rule-firing sequence (*"Inferred $\mathit{Not}_{1,2,3}$ because row 1 already contains value 3"* $\implies$ *"Deduce $\mathit{Is}_{1,2,4}$ as the last remaining candidate"*), or plain-English sentences explaining each elimination by row, column, or box constraint.

Import `atom`, `build_definite_kb`, `build_general_kb`, `solve_full_grid_fc`, `solve_full_grid_bc`, and `pl_bc_entails` from `sudoku_solver.py`. Do not copy or rewrite these core functions in the Streamlit file. You may add app-specific helper functions where needed for the interface or reasoning trace.

Refer to the provided `StreamlitDeploymentGuide.pdf` guide to deploy your code and include the link for your Streamlit app below.


### Streamlit implementation note

The deployed app imports the required core functions from `sudoku_solver.py`. Full-grid solving supports both FC and recursive BC. The separate entailment module calls `pl_bc_entails` directly and displays only the boolean verdict (`True` or `False`) after the user submits a query. Tutor mode instruments forward chaining independently over the whole solve: it records each newly inferred cell value in rule-firing order, shows cumulative board states, highlights the newest deduction, and presents the complete trace plus individual steps in collapsible panels.


## Submission

Submit only the following three files:

1. `Sudoku_Assignment.ipynb` — with all required cells run, outputs visible, and all conceptual questions answered.
2. `sudoku_solver.py` — containing your implementation of the knowledge-base and inference functions.
3. `sudoku_app.py` — containing your Streamlit application.

The provided support files are required to run the assignment but are not part of the student submission.

### Deployed Streamlit app URL

Paste your deployed Streamlit app URL here:

https://sudokuapppy-cvr6xwcrugb56jel75jmtw.streamlit.app/
